# An unconditional model with a fixed reference

A fixed reference encoder carries no trainable encoder state. It pins the input
state, and the diagonal learns everything else. The library represents an
unconditional model this way, as the one-condition case of a conditional one.

The reference here samples uniformly from even-parity bitstrings and never
changes during training. Its amplitude ratio is one for masks that preserve
parity and zero otherwise, and its circuit is a GHZ-style `RY(pi/2)` followed by
a CNOT ladder, so the simulation bridge can prepare it.

Even-parity support induces an exact property. Flipping every output bit is the
same as applying `X^n` before the final Hadamard layer, and `X^n H^n = H^n Z^n`,
so the flip only adds `Z^n` on the reference and a sign flip on the odd-weight
generator angles. The reference lives on even parity, so `Z^n` leaves it
unchanged, and `Ansatz.complete(n)` has no odd-weight generators. The model
distribution is therefore invariant under complementing every output bit, for
every parameter vector. The last cells measure that residual instead of
trusting the argument.


## Setup

The register holds 8 qubits, so the exact checks can enumerate all 256 basis
states and build the target law state by state. The budgets are small for the
same reason.


In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import pennylane as qml

import iqp_cond as ciqp

N_QUBITS = 8
# Per-qubit excitation probability of the target mixture.
BIAS = 0.12
N_TARGET, N_VALIDATION = 2000, 1000

## Define the fixed reference

`sample` draws `n - 1` uniform bits and appends their parity, so every
sampled bitstring has even weight. `amplitude_ratios` keeps a mask only when
it also has even weight, because an odd-weight mask would carry an
even-parity string to an odd-parity one and the ratio would be zero.

`preparation` returns a callable that queues the PennyLane gates preparing
the GHZ state, which is what the bridge runs before the first Hadamard layer.
The bridge queues whatever the callable queues, so the state can use any
PennyLane operation or template, not a fixed gate list. Training never calls
`preparation`.

`select` returns the encoder itself. There is one condition and no
per-condition state to slice.

The first line of the next cell keeps annotations lazy, since `select`
returns the class itself and that name only exists after the class body has
run.


In [2]:
from __future__ import annotations

from collections.abc import Callable
from dataclasses import dataclass


@dataclass(frozen=True)
class FixedParityEncoder:
    """One-condition parity reference with no trainable encoder state."""

    n_qubits: int

    @property
    def n_conditions(self) -> int:
        return 1

    def sample(self, key: jax.Array, n_samples: int) -> jnp.ndarray:
        # Draw n - 1 free bits and append their parity to force even weight.
        bits = jax.random.bernoulli(
            key, 0.5, shape=(n_samples, self.n_qubits - 1)
        ).astype(jnp.float64)
        parity = jnp.mod(jnp.sum(bits, axis=-1), 2.0)
        return jnp.concatenate((bits, parity[:, None]), axis=-1)[None, ...]

    def amplitude_ratios(self, samples: jnp.ndarray, masks: jnp.ndarray) -> jnp.ndarray:
        # Only even-weight masks preserve parity, and every ratio is one.
        even_masks = jnp.mod(jnp.sum(masks, axis=-1), 2.0) == 0
        return jnp.broadcast_to(
            even_masks[None, None, :],
            (1, samples.shape[1], masks.shape[0]),
        ).astype(jnp.float64)

    def preparation(self, condition: int) -> Callable[[], None]:
        if condition != 0:
            raise IndexError("the unconditional encoder has only condition 0")

        def queue() -> None:
            # RY(pi/2) on the first wire, then a CNOT ladder, gives the GHZ state.
            qml.RY(np.pi / 2.0, wires=0)
            for wire in range(1, self.n_qubits):
                qml.CNOT(wires=(wire - 1, wire))

        return queue

    def select(self, condition: int) -> FixedParityEncoder:
        if condition != 0:
            raise IndexError("the unconditional encoder has only condition 0")
        return self

## Build a complement-symmetric target

The target has to share the induced symmetry, or no parameter vector could fit
it. `complement_symmetric_target` draws each sample from `Bernoulli(p)` or from
`Bernoulli(1 - p)` with equal probability, so a bitstring and its complement are
equally likely.


In [3]:
def complement_symmetric_target(
    rng: np.random.Generator,
    probabilities: np.ndarray,
    n_samples: int,
) -> np.ndarray:
    """Draw an equal mixture of Bernoulli(p) and Bernoulli(1-p)."""
    component = rng.integers(0, 2, size=n_samples)
    # Per sample, either keep p or invert it, elementwise over the qubits.
    p = np.where(component[:, None] == 0, probabilities, 1.0 - probabilities)
    return (rng.random((n_samples, probabilities.size)) < p).astype(float)

### The exact target law

The same mixture written per state gives the exact probabilities to score
against. This quantity is complement-symmetric by construction, so it doubles as
a baseline for the symmetry check on the model.


In [4]:
# Enumerate every basis state and read its bits from the integer.
states = np.arange(2**N_QUBITS)
bits = ((states[:, None] >> np.arange(N_QUBITS)) & 1).astype(float)
component = np.full(N_QUBITS, BIAS)
# Equal mixture of the product law with bias p and with bias 1 - p.
target_probabilities = 0.5 * (
    np.prod(np.where(bits, component, 1.0 - component), axis=1)
    + np.prod(np.where(bits, 1.0 - component, component), axis=1)
)
# XOR with the all-ones mask gives the bitwise complement of each state.
complements = states ^ (2**N_QUBITS - 1)
target_error = np.max(np.abs(target_probabilities - target_probabilities[complements]))
print(f"exact target law over {states.size} states")
print(f"target complement-symmetry error {target_error:.3e}")

exact target law over 256 states
target complement-symmetry error 0.000e+00


## Train

An unconditional model takes a single condition, so the targets carry a leading
axis of length one. Two bandwidths are configured explicitly, and validation
uses a larger mask and Monte Carlo budget than training, which makes checkpoint
selection less noisy than the training signal.


In [5]:
rng = np.random.default_rng(12)
target = complement_symmetric_target(rng, component, N_TARGET)
validation = complement_symmetric_target(rng, component, N_VALIDATION)
model = ciqp.ConditionalModel(
    ciqp.Ansatz.complete(N_QUBITS), FixedParityEncoder(N_QUBITS)
)
config = ciqp.TrainingConfig(
    steps=150,
    batch_size=128,
    # Two Hamming-RBF bandwidths, kept fixed for this run.
    sigmas=(0.6, 1.2),
    n_ops=128,
    mc_samples=512,
    # Larger validation budgets, used only for checkpoint selection.
    validation_n_ops=256,
    validation_mc_samples=1000,
)
result = ciqp.train(
    model,
    # The leading axis holds the single condition.
    target[None, ...],
    validation[None, ...],
    seed=9,
    config=config,
)
print(f"initial validation MMD {result.validation_losses[0]:.6f}")
print(f"best validation MMD {result.best_validation_loss:.6f}")

initial validation MMD 0.019349
best validation MMD 0.001489


## Check the exact distribution

`model.probabilities` returns the exact visible distribution, so both the
distance to the target law and the residual symmetry error are measurable
directly. The symmetry error sits at machine precision for any parameter vector,
trained or not, because it comes from the reference rather than from the fit.


In [6]:
# make_qscript records the preparation without a device: one RY on the first
# wire, then the CNOT ladder, which is n_qubits operations in total.
script = qml.tape.make_qscript(model.encoder.preparation(0))()
print(f"queued preparation operations {len(script.operations)}")

probabilities = np.asarray(model.probabilities(result.best_params))[0]
# Total variation distance between the model and the exact target law.
target_tvd = 0.5 * np.abs(probabilities - target_probabilities).sum()
# How far the model is from its own complement relabeling.
complement_error = np.max(np.abs(probabilities - probabilities[complements]))
print(f"exact target TVD {target_tvd:.6f}")
print(f"maximum complement-symmetry error {complement_error:.3e}")

queued preparation operations 8
exact target TVD 0.086261
maximum complement-symmetry error 5.551e-17


## Where to go next

The fixed-reference pattern works for any classically tractable input state with
a known sampling law and amplitude ratio, which is how the library encodes a
structural constraint on the output distribution.

These exact checks need a register small enough to enumerate. Beyond that, the
MMD estimate is the only score available, and `examples/train_gaussian.ipynb`
shows the sampling path for larger registers.
